# GRA: nested learning

project = ```GRA```, host = ```mach```, device = ```cuda:0```

## Goals:

nested learning

In [1]:
# HIDE CODE


project_name = '_GRA'


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, project_name))
from utils.plotting import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

In [2]:
device_idx = 0
device = f'cuda:{device_idx}'

print(f"device: {device}  ———  host: {os.uname().nodename}")

device: cuda:0  ———  host: mach

## Main

In [3]:
from optimizers import *
from test.test import *

In [4]:
import torch
from torch.optim import Optimizer

class GRA(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), 
                 k=1.0, e_ref=1e-4, alpha_z=0.01, eps=1e-8):
        """
        General Relativistic Adaptation (GRA) Optimizer.
        
        Args:
            lr (float): Base learning rate (max speed at surface).
            betas (tuple): (beta1, beta2) for momentum and volatility.
            k (float): Curvature scale of the AdS space.
            e_ref (float): Reference energy threshold for consolidation.
            alpha_z (float): Relaxation rate for depth redistribution.
        """
        defaults = dict(lr=lr, betas=betas, k=k, e_ref=e_ref, 
                        alpha_z=alpha_z, eps=eps)
        super(GRA, self).__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            # Hyperparameters
            lr = group['lr']
            beta1, beta2 = group['betas']
            k = group['k']
            e_ref = group['e_ref']
            alpha_z = group['alpha_z']
            eps = group['eps']

            for p in group['params']:
                if p.grad is None:
                    continue
                
                grad = p.grad
                state = self.state[p]

                # Initialization
                if len(state) == 0:
                    state['step'] = 0
                    state['m'] = torch.zeros_like(p)     # Momentum
                    state['v'] = torch.zeros_like(p)     # Volatility
                    state['z'] = torch.ones_like(p)      # Depth (Start slightly deep)

                m, v, z = state['m'], state['v'], state['z']
                state['step'] += 1

                # 1. Update Momentum (Force Integration)
                m.mul_(beta1).add_(grad, alpha=1 - beta1)

                # 2. Update Volatility (Kinetic Energy)
                # We use m^2 as the proxy for kinetic energy/volatility
                v.mul_(beta2).addcmul_(m, m, value=1 - beta2)

                # 3. Compute Equilibrium Depth (Buoyancy Balance)
                # z_target = (1/2k) * ln( E_ref / (v + eps) )
                # If v is high (hot), ratio is small, ln is negative -> z goes to 0 (Surface)
                # If v is low (cold), ratio is huge, ln is positive -> z goes to infinity (Bulk)
                energy_ratio = e_ref / (v + eps)
                z_target = (torch.log(energy_ratio) / (2 * k)).clamp_(min=0.0)

                # 4. Relax Depth (Redistribution)
                z.mul_(1 - alpha_z).add_(z_target, alpha=alpha_z)

                # 5. Compute Warp Factor (Gravitational Redshift)
                # Effective LR = Base LR * e^(-2kz)
                warp_factor = torch.exp(-2 * k * z)

                # 6. Update Weights
                p.addcmul_(m, warp_factor, value=-lr)

        return loss

In [14]:
class Dynamic_AdS_Optimizer(torch.optim.Optimizer):
    def __init__(self, params, base_lr=1e-3, gravity=1e-5):
        # gravity: The constant force pulling params into the deep bulk (consolidation)
        defaults = dict(base_lr=base_lr, gravity=gravity)
        super().__init__(params, defaults)

        for group in self.param_groups:
            for p in group['params']:
                state = self.state[p]
                # Initialize z at some median depth (e.g., z=1.0)
                state['z'] = torch.ones_like(p.data) * 1.0

    def step(self):
        for group in self.param_groups:
            lr = group['base_lr']
            gravity = group['gravity']
            
            for p in group['params']:
                if p.grad is None: continue
                state = self.state[p]
                
                # 1. Calculate the Learning Update (Velocity)
                # v_theta = - lr * exp(-z) * grad
                time_dilation = torch.exp(-state['z'])
                update_step = -lr * time_dilation * p.grad.data
                
                # 2. Update the Parameter (Theta)
                p.data.add_(update_step)
                
                # 3. Update the Depth (z) - The Redistribution
                # Force Up = Magnitude of update (volatility)
                # Force Down = Gravity constant
                # z_new = z_old - (update_magnitude - gravity)
                
                # We normalize update_magnitude so it's comparable to gravity
                update_mag = torch.abs(update_step) 
                
                # If update is large, z decreases (floats up).
                # If update is small, z increases (sinks down).
                delta_z = - (update_mag - gravity)
                
                state['z'].add_(delta_z)
                
                # Clamp z to avoid numerical explosions (z > 0)
                state['z'].clamp_(min=0.0, max=10.0)

In [11]:
# 1. The Environment: Polyrhythmic Drift
class PolyrhythmicTeacher:
    def __init__(self, n_features, max_period=1000):
        self.n = n_features
        # Frequencies distributed on a log scale (Power Law)
        # From very fast (period=2) to very slow (period=max_period)
        self.periods = np.logspace(np.log10(2), np.log10(max_period), n_features)
        self.phases = np.random.rand(n_features) * 2 * np.pi
        
    def get_target_weights(self, t):
        # w*(t) = sin(t * 2pi / period + phase)
        return torch.tensor([np.sin(t * 2 * np.pi / p + ph) 
                             for p, ph in zip(self.periods, self.phases)]).float()


# 2. The GRA Optimizer (AdS-Optimizer)
class AdS_SGD(torch.optim.Optimizer):
    def __init__(self, params, base_lr=0.01, sigma=2.0):
        defaults = dict(base_lr=base_lr, sigma=sigma)
        super().__init__(params, defaults)
        
        for group in self.param_groups:
            for p in group['params']:
                # The "Geometry": Sample depth z from Normal(0, sigma)
                # Learning Rate = base_lr * exp(-|z|) -> Log-Normal Distribution
                z = torch.randn_like(p.data) * group['sigma']
                # Heavy tail distribution of learning rates
                state = self.state[p]
                state['lr_mask'] = torch.exp(-torch.abs(z)).to(p.device)
                
    def step(self):
        for group in self.param_groups:
            lr = group['base_lr']
            for p in group['params']:
                if p.grad is None: continue
                # Update: w = w - lr * N(z) * grad
                p.data.add_(-lr * self.state[p]['lr_mask'] * p.grad.data)

            
# 3. The Experiment
def run_experiment(optimizer_type='ads', steps=2000):
    n_features = 100
    teacher = PolyrhythmicTeacher(n_features)
    
    # Linear model: y = w * x
    model = nn.Linear(n_features, 1, bias=False)
    
    if optimizer_type == 'ads':
        opt = GRA(model.parameters(), lr=0.001)
    elif optimizer_type == 'sgd':
        opt = torch.optim.SGD(model.parameters(), lr=0.001)
    elif optimizer_type == 'adam':
        opt = torch.optim.Adam(model.parameters(), lr=0.001)

    errors = []
    
    for t in range(steps):
        target_w = teacher.get_target_weights(t)
        x = torch.randn(n_features)
        
        # True target
        y_true = torch.dot(target_w, x)
        
        # Prediction
        y_pred = model(x)
        
        loss = (y_pred - y_true)**2
        
        opt.zero_grad()
        loss.backward()
        opt.step()
        
        errors.append(loss.item())
        
    return errors

In [14]:
# Compare
loss_gra = run_experiment('ads', steps=2_000)
loss_sgd = run_experiment('sgd', steps=2_000)
loss_adam = run_experiment('adam', steps=2_000)

# Visualization (Mental check: GRA should be lower and smoother)
# GRA adapts to the 'slow' weights without oscillating on the 'fast' ones.
print(f"Final Loss GRA: {np.mean(loss_gra[-100:]):.4f}")
print(f"Final Loss SGD: {np.mean(loss_sgd[-100:]):.4f}")
print(f"Final Loss Adam: {np.mean(loss_adam[-100:]):.4f}")

Final Loss GRA: 58.5548

Final Loss SGD: 64.6849

Final Loss Adam: 41.0014